# LeetCode 2140 — Solving Questions With Brainpower bằng Python

Link đề: https://leetcode.com/problems/solving-questions-with-brainpower/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def mostPoints(self, questions: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận dạng bài **quy hoạch động "chọn hoặc bỏ"** (take / skip) giống House Robber nhưng bước nhảy **thay đổi theo từng phần tử**.
2. Định nghĩa trạng thái đúng: `dp[i]` = điểm tối đa **từ câu `i` tới hết** (DP hậu tố) — và hiểu vì sao DP **tiền tố** kiểu "kéo" (pull) khó viết hơn ở bài này.
3. Đi từ **vét cạn 2ⁿ** → **đệ quy có nhớ** (và bẫy giới hạn đệ quy với n = 10⁵) → **DP lặp từ phải sang trái** → **DP "đẩy" (push) từ trái sang phải**.
4. **Truy vết** để in ra danh sách câu được chọn.
5. Viết test đối chiếu ngẫu nhiên với vét cạn, kiểm tra cả "tính hợp lệ" của tập câu đã chọn.

## 2. Đề bài & ví dụ

Mảng `questions[i] = [points_i, brainpower_i]` mô tả các câu hỏi của bài thi, phải xét **theo thứ tự** từ câu 0. Với mỗi câu, quyết định **giải** hoặc **bỏ qua**:
- **Giải** câu `i`: được `points_i` điểm, nhưng **không được giải** `brainpower_i` câu **kế tiếp** (bị bỏ qua bắt buộc).
- **Bỏ qua** câu `i`: chuyển sang quyết định câu `i + 1`.

Trả về **tổng điểm lớn nhất**.

**Ví dụ 1:** `[[3,2],[4,3],[4,4],[2,5]]` → `5` (giải câu 0 (+3, khóa câu 1, 2), giải câu 3 (+2)).

**Ví dụ 2:** `[[1,1],[2,2],[3,3],[4,4],[5,5]]` → `7` (bỏ câu 0, giải câu 1 (+2, khóa 2, 3), giải câu 4 (+5)).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ points_i, brainpower_i ≤ 10⁵`.

In [1]:
from typing import List
from functools import lru_cache
from itertools import product
import random, time, sys

examples = [
    ([[3, 2], [4, 3], [4, 4], [2, 5]], 5),
    ([[1, 1], [2, 2], [3, 3], [4, 4], [5, 5]], 7),
]
def score(questions, chosen):
    # kiểm tra tập câu chọn có hợp lệ không, trả về tổng điểm (hoặc None nếu vi phạm)
    blocked_until = -1; total = 0
    for i in sorted(chosen):
        if i <= blocked_until: return None
        p, b = questions[i]; total += p; blocked_until = i + b
    return total

print(score(examples[0][0], [0, 3]), score(examples[0][0], [0, 1]), score(examples[1][0], [1, 4]))

5 None 7


## 3. Kiến thức nền

### 3.1 Quy hoạch động: trạng thái + chuyển trạng thái + thứ tự tính

Ba câu hỏi luôn cần trả lời:
1. **Trạng thái**: `dp[i]` nghĩa là gì?
2. **Chuyển**: `dp[i]` tính từ những trạng thái nào?
3. **Thứ tự**: tính `dp` theo chiều nào để khi cần thì giá trị đã có sẵn?

Ví dụ khởi động — House Robber (không được lấy hai nhà liền kề): `dp[i] = max(dp[i+1], a[i] + dp[i+2])`.

In [2]:
def house_robber(a):
    n = len(a); dp = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        dp[i] = max(dp[i + 1], a[i] + dp[i + 2])
    return dp[0]
print(house_robber([2, 7, 9, 3, 1]))   # 2 + 9 + 1 = 12

12


Bài 2140 là House Robber với "bước nhảy sau khi lấy" = `brainpower_i + 1` thay vì 2.

### 3.2 `functools.lru_cache` / `cache` và giới hạn đệ quy

Đệ quy có nhớ viết rất tự nhiên, nhưng độ sâu đệ quy của `dfs(0) → dfs(1) → … → dfs(n)` có thể là `n = 10⁵` — vượt giới hạn mặc định 1000 của CPython. Trên LeetCode lời giải đệ quy vẫn hay qua vì họ nâng giới hạn, nhưng **không nên dựa vào điều đó**.

In [3]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
@lru_cache(None)
def depth(i):
    return 0 if i == 0 else 1 + depth(i - 1)
try:
    depth(5000)
except RecursionError as e:
    print("depth(5000) -> RecursionError:", str(e)[:40])

giới hạn đệ quy mặc định: 1000
depth(5000) -> RecursionError: maximum recursion depth exceeded


### 3.3 `min(n, j)` để chặn chỉ số vượt mảng

Sau khi giải câu `i`, câu kế tiếp có thể xét là `j = i + b + 1`, có thể vượt `n`. Dùng mảng `dp` dài `n + 1` với `dp[n] = 0` và chặn `min(n, j)`.

In [4]:
n = 4
for i, b in [(0, 2), (1, 3), (3, 5)]:
    print(f"giải câu {i} (b={b}) -> câu tiếp theo được xét: {i + b + 1} -> chặn: {min(n, i + b + 1)}")

giải câu 0 (b=2) -> câu tiếp theo được xét: 3 -> chặn: 3
giải câu 1 (b=3) -> câu tiếp theo được xét: 5 -> chặn: 4
giải câu 3 (b=5) -> câu tiếp theo được xét: 9 -> chặn: 4


### 3.4 Liệt kê mọi tập con bằng `itertools.product([0, 1], repeat=n)`

Dùng cho vét cạn kiểm chứng.

In [5]:
print(list(product([0, 1], repeat=3))[:5], "...", 2 ** 3, "tập con")

[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0)] ... 8 tập con


## 4. Ý tưởng nền tảng / trực giác

Đứng ở câu `i`, chỉ có hai lựa chọn — và **tương lai chỉ phụ thuộc vào vị trí** câu tiếp theo được xét:
- **Bỏ**: tương lai bắt đầu ở `i + 1`.
- **Giải**: được `p_i`, tương lai bắt đầu ở `i + b_i + 1`.

Vì vậy "điểm tối đa từ câu `i` trở đi" chỉ phụ thuộc `i` ⇒ trạng thái `dp[i]`:

`dp[i] = max(dp[i + 1], p_i + dp[min(n, i + b_i + 1)])`, `dp[n] = 0`.

Tính tay ví dụ 1 (`n = 4`) **từ phải sang trái**:

| i | (p, b) | bỏ: dp[i+1] | giải: p + dp[min(4, i+b+1)] | dp[i] |
|---|---|---|---|---|
| 4 | — | — | — | 0 |
| 3 | (2, 5) | dp[4] = 0 | 2 + dp[4] = 2 | 2 |
| 2 | (4, 4) | dp[3] = 2 | 4 + dp[4] = 4 | 4 |
| 1 | (4, 3) | dp[2] = 4 | 4 + dp[4] = 4 | 4 |
| 0 | (3, 2) | dp[1] = 4 | 3 + dp[3] = 5 | **5** |

Vì sao **tham lam** không được? "Luôn giải câu điểm cao nhất" ở ví dụ 1 sẽ chọn câu 1 hoặc 2 (4 điểm) nhưng chặn mất câu 3 ⇒ chỉ 4 < 5.

In [6]:
q = examples[0][0]; n = len(q); dp = [0] * (n + 1)
for i in range(n - 1, -1, -1):
    p, b = q[i]
    skip, take = dp[i + 1], p + dp[min(n, i + b + 1)]
    dp[i] = max(skip, take)
    print(f"i={i} (p={p}, b={b}): bỏ={skip}, giải={take} -> dp[{i}]={dp[i]}")
print("dp =", dp)

i=3 (p=2, b=5): bỏ=0, giải=2 -> dp[3]=2
i=2 (p=4, b=4): bỏ=2, giải=4 -> dp[2]=4
i=1 (p=4, b=3): bỏ=4, giải=4 -> dp[1]=4
i=0 (p=3, b=2): bỏ=4, giải=5 -> dp[0]=5
dp = [5, 4, 4, 2, 0]


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi chuỗi quyết định (đệ quy không nhớ)

In [7]:
def brute_recursive(questions):
    n = len(questions)
    def go(i):
        if i >= n:
            return 0
        p, b = questions[i]
        return max(go(i + 1), p + go(i + b + 1))
    return go(0)

for q, exp in examples:
    got = brute_recursive(q)
    print(got, "OK" if got == exp else "SAI")

5 OK
7 OK


**Độ phức tạp:** mỗi lời gọi tách hai nhánh ⇒ tới O(2ⁿ) (khi `b` nhỏ). Lời gọi `go(i)` với cùng `i` bị tính **lặp đi lặp lại** — đó là nút thắt (bài toán con chồng lấp).

### 5.2 Vét cạn mọi tập con + kiểm tra hợp lệ (độc lập hoàn toàn với công thức DP)

Dùng làm "chuẩn vàng" để kiểm chứng: thử cả `2ⁿ` tập con, giữ những tập hợp lệ theo đúng luật của đề.

In [8]:
def brute_subsets(questions):
    n = len(questions); best = 0
    for mask in product([0, 1], repeat=n):
        chosen = [i for i in range(n) if mask[i]]
        s = score(questions, chosen)
        if s is not None and s > best: best = s
    return best

for q, exp in examples:
    print(brute_subsets(q), "OK" if brute_subsets(q) == exp else "SAI")

5 OK
7 OK


In [9]:
def count_calls(questions):
    n = len(questions); calls = [0]
    def go(i):
        calls[0] += 1
        if i >= n: return 0
        p, b = questions[i]
        return max(go(i + 1), p + go(i + b + 1))
    go(0); return calls[0]

for n in [10, 15, 20]:
    print(f"n={n}, mọi b=1: số lời gọi đệ quy = {count_calls([[1, 1]] * n)}")

n=10, mọi b=1: số lời gọi đệ quy = 287
n=15, mọi b=1: số lời gọi đệ quy = 3193
n=20, mọi b=1: số lời gọi đệ quy = 35421


Số lời gọi tăng theo dãy Fibonacci (~1,6ⁿ). Ghi nhớ (memo) cắt xuống còn `n + 1` trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "Giải/bỏ câu `i`" chỉ ảnh hưởng tương lai qua **chỉ số câu tiếp theo được xét** ⇒ bài toán con `dp[i]` = "điểm tối đa trên hậu tố bắt đầu ở `i`" có **cấu trúc con tối ưu**:

`dp[i] = max(dp[i + 1], p_i + dp[min(n, i + b_i + 1)])`.

Vì `dp[i]` phụ thuộc các chỉ số **lớn hơn** `i` ⇒ tính **từ phải sang trái**.

**Quy tắc:**
- **Nếu** `i + b_i + 1 ≥ n` **thì** giải câu `i` là câu cuối cùng được giải ⇒ `take = p_i`.
- **Nếu** bỏ tốt hơn **thì** `dp[i] = dp[i + 1]` (đáp án không giảm khi đi sang trái: `dp` là dãy **không tăng** theo `i`).

**Vì sao DP tiền tố "kéo" (pull) khó?** Muốn `best[i]` = "điểm tối đa dùng các câu `< i` sao cho câu `i` còn được xét" thì phải biết **mọi** câu `j < i` có `j + b_j + 1 ≤ i` — mỗi `i` có thể có nhiều `j` nhảy tới ⇒ dễ viết sai. Cách thuận tay hơn khi đi xuôi là **đẩy** (push): từ `i` cập nhật sang `i + 1` và `i + b_i + 1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đệ quy có nhớ (top-down)

In [10]:
def memo_topdown(questions):
    n = len(questions)
    @lru_cache(None)
    def dfs(i):
        if i >= n:
            return 0
        p, b = questions[i]
        return max(dfs(i + 1), p + dfs(i + b + 1))
    return dfs(0)

for q, exp in examples:
    print(memo_topdown(q), "OK" if memo_topdown(q) == exp else "SAI")

5 OK
7 OK


**Độ phức tạp:** O(n) thời gian và bộ nhớ; nhưng độ sâu đệ quy O(n) ⇒ với `n = 10⁵` có nguy cơ `RecursionError` (mục 13). Nếu muốn dùng trên LeetCode thì cần `sys.setrecursionlimit` lớn — không khuyến khích.

### 7.2 Cách 3 — DP lặp từ phải sang trái (bottom-up) — cách chuẩn

In [11]:
def dp_backward(questions):
    n = len(questions)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        p, b = questions[i]
        dp[i] = max(dp[i + 1], p + dp[min(n, i + b + 1)])
    return dp[0]

for q, exp in examples:
    print(dp_backward(q), "OK" if dp_backward(q) == exp else "SAI")

5 OK
7 OK


**Độ phức tạp cách 3:** O(n) thời gian, O(n) bộ nhớ. Không đệ quy, không lo giới hạn.

Không thể giảm xuống O(1) bộ nhớ như House Robber vì bước nhảy `b_i` có thể rất xa — cần giữ cả mảng.

### 7.3 Cách 4 — DP "đẩy" từ trái sang phải (forward / push)

`f[i]` = điểm tối đa **đã tích lũy** khi ta **đứng trước** câu `i` và được tự do quyết định câu `i`. Từ `i`:
- bỏ: `f[i + 1] = max(f[i + 1], f[i])`;
- giải: `f[j] = max(f[j], f[i] + p_i)` với `j = min(n, i + b_i + 1)`.

Đáp án `f[n]`. Vì mọi cạnh đi từ chỉ số nhỏ sang lớn, duyệt `i` tăng dần là đúng thứ tự.

In [12]:
def dp_forward(questions):
    n = len(questions)
    f = [0] * (n + 1)
    for i, (p, b) in enumerate(questions):
        f[i + 1] = max(f[i + 1], f[i])
        j = min(n, i + b + 1)
        f[j] = max(f[j], f[i] + p)
    return f[n]

for q, exp in examples:
    print(dp_forward(q), "OK" if dp_forward(q) == exp else "SAI")

5 OK
7 OK


Vì sao `f[i]` đã "chốt" khi ta xử lý `i`? Mọi cập nhật vào `f[i]` đến từ các `k < i` (bỏ từ `i − 1`, hoặc giải từ `k` với `k + b_k + 1 = i` hoặc bị chặn về `n`), và chúng đã được xử lý trước.

### 7.4 Cách 5 — DP lặp + **truy vết** câu được chọn

In [13]:
def dp_with_choice(questions):
    n = len(questions); dp = [0] * (n + 1); take = [False] * n
    for i in range(n - 1, -1, -1):
        p, b = questions[i]
        t = p + dp[min(n, i + b + 1)]
        if t > dp[i + 1]:
            dp[i] = t; take[i] = True
        else:
            dp[i] = dp[i + 1]
    chosen = []; i = 0
    while i < n:
        if take[i]:
            chosen.append(i); i += questions[i][1] + 1
        else:
            i += 1
    return dp[0], chosen

for q, exp in examples:
    best, ch = dp_with_choice(q)
    print(best, ch, "| kiểm tra score:", score(q, ch))

5 [0, 3] | kiểm tra score: 5
7 [1, 4] | kiểm tra score: 7


Khi truy vết, ta đi **xuôi** từ câu 0 và tuân theo đúng quyết định đã lưu: nếu `take[i]` thì nhảy `b_i + 1`, ngược lại sang `i + 1`. Tập thu được luôn hợp lệ và có tổng = `dp[0]`.

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên ví dụ 2: `[[1,1],[2,2],[3,3],[4,4],[5,5]]`

| i | (p, b) | bỏ = dp[i+1] | giải = p + dp[min(5, i+b+1)] | dp[i] |
|---|---|---|---|---|
| 4 | (5, 5) | 0 | 5 + dp[5] = 5 | 5 |
| 3 | (4, 4) | 5 | 4 + dp[5] = 4 | 5 |
| 2 | (3, 3) | 5 | 3 + dp[5] = 3 | 5 |
| 1 | (2, 2) | 5 | 2 + dp[4] = 7 | 7 |
| 0 | (1, 1) | 7 | 1 + dp[2] = 6 | 7 |

In [14]:
def trace_backward(questions):
    n = len(questions); dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        p, b = questions[i]; j = min(n, i + b + 1)
        skip, take = dp[i + 1], p + dp[j]
        dp[i] = max(skip, take)
        print(f"i={i} (p={p}, b={b}): bỏ = dp[{i+1}] = {skip}; giải = {p} + dp[{j}] = {take} -> dp[{i}] = {dp[i]}"
              f" ({'giải' if take > skip else 'bỏ'})")
    print("dp =", dp)
    return dp[0]

trace_backward(examples[1][0])

i=4 (p=5, b=5): bỏ = dp[5] = 0; giải = 5 + dp[5] = 5 -> dp[4] = 5 (giải)
i=3 (p=4, b=4): bỏ = dp[4] = 5; giải = 4 + dp[5] = 4 -> dp[3] = 5 (bỏ)
i=2 (p=3, b=3): bỏ = dp[3] = 5; giải = 3 + dp[5] = 3 -> dp[2] = 5 (bỏ)
i=1 (p=2, b=2): bỏ = dp[2] = 5; giải = 2 + dp[4] = 7 -> dp[1] = 7 (giải)
i=0 (p=1, b=1): bỏ = dp[1] = 7; giải = 1 + dp[2] = 6 -> dp[0] = 7 (bỏ)
dp = [7, 7, 5, 5, 5, 0]


7

### 8.2 Trace cách 4 (đẩy) trên ví dụ 1

In [15]:
def trace_forward(questions):
    n = len(questions); f = [0] * (n + 1)
    for i, (p, b) in enumerate(questions):
        j = min(n, i + b + 1)
        old1, oldj = f[i + 1], f[j]
        f[i + 1] = max(f[i + 1], f[i])
        f[j] = max(f[j], f[i] + p)
        print(f"i={i}: f[i]={f[i]} | bỏ: f[{i+1}] {old1}->{f[i+1]} | giải: f[{j}] {oldj}->{f[j]} | f={f}")
    return f[n]

print("đáp án:", trace_forward(examples[0][0]))

i=0: f[i]=0 | bỏ: f[1] 0->0 | giải: f[3] 0->3 | f=[0, 0, 0, 3, 0]
i=1: f[i]=0 | bỏ: f[2] 0->0 | giải: f[4] 0->4 | f=[0, 0, 0, 3, 4]
i=2: f[i]=0 | bỏ: f[3] 3->3 | giải: f[4] 4->4 | f=[0, 0, 0, 3, 4]
i=3: f[i]=3 | bỏ: f[4] 4->5 | giải: f[4] 4->5 | f=[0, 0, 0, 3, 5]
đáp án: 5


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def mostPoints(self, questions: List[List[int]]) -> int:
        n = len(questions)
        dp = [0] * (n + 1)                 # dp[i] = điểm tối đa từ câu i tới hết; dp[n] = 0
        for i in range(n - 1, -1, -1):
            p, b = questions[i]
            nxt = i + b + 1
            take = p + (dp[nxt] if nxt < n else 0)
            dp[i] = max(dp[i + 1], take)
        return dp[0]

In [17]:
sol = Solution()
for q, exp in examples:
    got = sol.mostPoints(q)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

5 | kỳ vọng 5 | OK
7 | kỳ vọng 7 | OK


## 10. Vì sao thuật toán đúng

**Mệnh đề:** với mọi `i ∈ [0, n]`, `dp[i]` = điểm tối đa có thể đạt được chỉ dùng các câu `i, i+1, …, n−1` (bắt đầu tự do ở câu `i`).

Chứng minh quy nạp lùi theo `i`:
- `i = n`: không còn câu ⇒ 0 = `dp[n]`. ✔
- Giả sử đúng với mọi chỉ số > `i`. Xét một cách chơi tối ưu trên hậu tố `i`:
  - Nếu nó **bỏ** câu `i`: phần còn lại là một cách chơi hợp lệ trên hậu tố `i + 1` ⇒ điểm ≤ `dp[i + 1]`.
  - Nếu nó **giải** câu `i`: các câu `i+1..i+b_i` bị khóa, phần còn lại là cách chơi hợp lệ trên hậu tố `min(n, i + b_i + 1)` ⇒ điểm ≤ `p_i + dp[min(n, i+b_i+1)]`.
  - Ngược lại, cả hai giá trị đều **đạt được** (ghép lựa chọn ở `i` với cách chơi tối ưu của hậu tố tương ứng).
  ⇒ `dp[i] = max(...)` đúng bằng giá trị tối ưu. ∎

Tính đúng của cách 4 (đẩy): `f[i]` = điểm tối đa dùng các câu `< i` sao cho câu `i` không bị khóa — mỗi trạng thái "đứng trước câu `i`" chỉ đến từ "bỏ `i − 1`" hoặc "giải `k` mà khóa kết thúc ngay trước `i`", đúng hai loại cạnh ta đẩy.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy không nhớ / tập con | O(2ⁿ) | O(n) | chỉ n ≤ 20 |
| 2. Đệ quy có nhớ | O(n) | O(n) + ngăn xếp O(n) | rủi ro RecursionError |
| 3. DP lùi | O(n) | O(n) | **chuẩn** |
| 4. DP đẩy | O(n) | O(n) | duyệt xuôi |
| 5. DP + truy vết | O(n) | O(n) | khi cần danh sách câu |

Đáp án lớn nhất ≈ `n · max(p)` = 10¹⁰ — vượt `int32` (C++/Java dùng `long long`/`long`).

In [18]:
rng = random.Random(2140)
big = [[rng.randint(1, 10**5), rng.randint(1, 10**5 if rng.random() < 0.3 else 5)] for _ in range(10**5)]
for name, f in [("dp_backward", dp_backward), ("dp_forward", dp_forward), ("Solution", Solution().mostPoints)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<12} -> {r}  {dt*1000:.0f} ms")
worst = [[10**5, 1]] * 10**5
print("điểm lớn nhất kiểu mọi câu (1e5, 1):", Solution().mostPoints(worst), "> 2^31 - 1 =", 2**31 - 1)

dp_backward  -> 1538986628  14 ms


dp_forward   -> 1538986628  21 ms


Solution     -> 1538986628  14 ms
điểm lớn nhất kiểu mọi câu (1e5, 1): 5000000000 > 2^31 - 1 = 2147483647


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute_recursive": brute_recursive, "brute_subsets": brute_subsets, "memo_topdown": memo_topdown,
       "dp_backward": dp_backward, "dp_forward": dp_forward,
       "dp_with_choice": lambda q: dp_with_choice(q)[0], "Solution": Solution().mostPoints}
edge = examples + [
    ([[5, 1]], 5),
    ([[1, 1], [1, 1]], 1),
    ([[1, 1], [100, 1]], 100),
    ([[10, 100], [1, 1], [1, 1]], 10),
    ([[1, 1], [1, 1], [1, 1], [1, 1]], 2),
    ([[21, 5], [92, 3], [74, 2], [39, 4], [58, 2], [5, 5], [49, 4], [65, 3]], 157),
]
for q, exp in edge:
    for name, f in ALL.items():
        assert f([list(x) for x in q]) == exp, (name, q, f(q), exp)
print("Edge case OK:", len(edge))

Edge case OK: 8


Case cuối `[[21,5],[92,3],[74,2],[39,4],[58,2],[5,5],[49,4],[65,3]]` (tự dựng — đối chiếu lại đề gốc nếu cần): các cách đều cho 157 (`92 + 65`), và brute force tập con xác nhận.

### 12.2 Đối chiếu ngẫu nhiên với vét cạn tập con + kiểm tra tập câu truy vết

In [20]:
rng = random.Random(42)
for _ in range(1500):
    n = rng.randint(1, 12)
    q = [[rng.randint(1, 20), rng.randint(1, rng.choice([1, 2, 4, 12]))] for _ in range(n)]
    ref = brute_subsets(q)
    for name, f in ALL.items():
        assert f(q) == ref, (name, q)
    best, ch = dp_with_choice(q)
    assert score(q, ch) == best == ref
print("1500 case ngẫu nhiên OK")

1500 case ngẫu nhiên OK


### 12.3 Tính chất: `dp` không tăng theo `i`; thêm câu vào **đầu** không làm giảm đáp án

In [21]:
rng = random.Random(7)
for _ in range(300):
    q = [[rng.randint(1, 50), rng.randint(1, 6)] for _ in range(rng.randint(1, 40))]
    n = len(q); dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        p, b = q[i]; dp[i] = max(dp[i + 1], p + dp[min(n, i + b + 1)])
    assert all(dp[i] >= dp[i + 1] for i in range(n))
    extra = [[rng.randint(1, 50), rng.randint(1, 6)]]
    assert Solution().mostPoints(extra + q) >= Solution().mostPoints(q)
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn giải câu có điểm cao nhất còn giải được

In [22]:
def wrong_greedy_max(questions):
    n = len(questions); blocked = [False] * n; total = 0
    order = sorted(range(n), key=lambda i: -questions[i][0])
    taken = []
    for i in order:
        # câu i giải được nếu không câu đã chọn nào khóa nó và nó không khóa câu đã chọn
        ok = all(not (j < i <= j + questions[j][1]) and not (i < j <= i + questions[i][1]) for j in taken)
        if ok: taken.append(i); total += questions[i][0]
    return total

print("bản sai :", wrong_greedy_max(examples[0][0]))
print("bản đúng:", Solution().mostPoints(examples[0][0]))

bản sai : 4
bản đúng: 5


**Vì sao sai:** câu 4 điểm khóa nhiều câu sau; tổng 3 + 2 tốt hơn. Bài có **tương tác** giữa các lựa chọn ⇒ cần DP.

### Lỗi 2 — Nhảy sai: `i + b` thay vì `i + b + 1`

In [23]:
def wrong_jump(questions):
    n = len(questions); dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        p, b = questions[i]
        dp[i] = max(dp[i + 1], p + dp[min(n, i + b)])       # SAI: câu i+b vẫn bị khóa
    return dp[0]

print("bản sai :", wrong_jump(examples[0][0]))
print("bản đúng:", Solution().mostPoints(examples[0][0]))

bản sai : 7
bản đúng: 5


**Vì sao sai:** giải câu `i` khóa các câu `i+1, …, i+b` (đúng `b` câu) ⇒ câu tự do tiếp theo là `i + b + 1`. Bản sai cho phép giải câu `i + b` ⇒ đếm thừa (ở đây ra 7: giải câu 0 rồi câu 2).

### Lỗi 3 — Quên chặn chỉ số: `IndexError`

In [24]:
def wrong_no_clamp(questions):
    n = len(questions); dp = [0] * (n + 1)
    try:
        for i in range(n - 1, -1, -1):
            p, b = questions[i]
            dp[i] = max(dp[i + 1], p + dp[i + b + 1])          # SAI: i+b+1 có thể > n
        return dp[0]
    except IndexError as e:
        return f"IndexError: {e}"

print("bản sai :", wrong_no_clamp(examples[0][0]))
print("bản đúng:", Solution().mostPoints(examples[0][0]))

bản sai : IndexError: list index out of range
bản đúng: 5


**Sửa:** `dp[min(n, i + b + 1)]` hoặc kiểm tra `nxt < n`.

### Lỗi 4 — Đệ quy có nhớ với `n` lớn: `RecursionError`

In [25]:
big_chain = [[1, 1]] * 20000
try:
    print(memo_topdown(big_chain))
except RecursionError as e:
    print("đệ quy có nhớ -> RecursionError:", str(e)[:45])
print("DP lặp:", Solution().mostPoints(big_chain))

đệ quy có nhớ -> RecursionError: maximum recursion depth exceeded
DP lặp: 10000


**Sửa:** dùng DP lặp (cách 3/4). Nếu bắt buộc dùng đệ quy, phải `sys.setrecursionlimit` rất lớn (và có thể tràn stack C) — không an toàn.

### Lỗi 5 — DP xuôi kiểu "kéo" chỉ nhìn câu liền trước

In [26]:
def wrong_forward_pull(questions):
    n = len(questions); best = [0] * (n + 1)          # best[i]: điểm tối đa dùng câu < i
    for i in range(n):
        p, b = questions[i]
        # SAI: chỉ cập nhật best[i+1] từ best[i] + p mà không tôn trọng việc khóa b câu
        best[i + 1] = max(best[i], (best[i - b] if i - b >= 0 else 0) + p)
    return best[n]

print("bản sai :", wrong_forward_pull(examples[0][0]))
print("bản đúng:", Solution().mostPoints(examples[0][0]))

bản sai : 4
bản đúng: 5


**Vì sao sai:** ràng buộc khóa là **về phía sau** của câu được giải (do `b_i` của chính nó), không phải về phía trước. Bản sai nhầm thành "câu `i` cần `b_i` câu trước trống" — đó là một bài khác. **Sửa:** DP hậu tố (cách 3) hoặc DP đẩy (cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Duyệt theo thứ tự, mỗi phần tử **lấy/bỏ**, lấy thì **mất quyền** với vài phần tử sau" ⇒ DP hậu tố `dp[i] = max(dp[i+1], gain + dp[next(i)])`.
- Khi ràng buộc hướng **về sau** của phần tử được chọn ⇒ DP **từ phải sang trái** là tự nhiên nhất; muốn đi xuôi thì dùng **đẩy**.
- Đệ quy có nhớ là cách nghĩ tốt, nhưng **nộp bản lặp** khi `n` tới 10⁵.
- Bài cùng họ: 198 House Robber, 1235 Maximum Profit in Job Scheduling (khoảng thời gian + nhị phân), 2008 Maximum Earnings From Taxi, 1751, 2054.

## 15. Biến thể của bài

### 15.1 Biến thể A — Giới hạn **số câu được giải tối đa `k`**

Thêm chiều số câu: `dp[i][c]` = điểm tối đa từ câu `i` khi còn được giải `c` câu. O(n·k).

In [27]:
def most_points_k(questions, k):
    n = len(questions)
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        p, b = questions[i]; j = min(n, i + b + 1)
        for c in range(k + 1):
            dp[i][c] = dp[i + 1][c]
            if c > 0:
                dp[i][c] = max(dp[i][c], p + dp[j][c - 1])
    return dp[0][k]

def most_points_k_brute(questions, k):
    n = len(questions); best = 0
    for mask in product([0, 1], repeat=n):
        ch = [i for i in range(n) if mask[i]]
        if len(ch) <= k:
            s = score(questions, ch)
            if s is not None: best = max(best, s)
    return best

print(most_points_k(examples[1][0], 1), most_points_k(examples[1][0], 2))
rng = random.Random(151)
for _ in range(400):
    q = [[rng.randint(1, 20), rng.randint(1, 3)] for _ in range(rng.randint(1, 9))]
    kk = rng.randint(0, 4)
    assert most_points_k(q, kk) == most_points_k_brute(q, kk)
print("khớp brute force")

5 7
khớp brute force


### 15.2 Biến thể B — Giải câu `i` khóa cả `b_i` câu **trước** nó (đối xứng)

Đảo ngược mảng và quay về bài gốc! Vì "khóa `b` câu trước" trên mảng gốc chính là "khóa `b` câu sau" trên mảng đảo.

Nhưng cẩn thận: nếu khóa **cả hai phía** thì không còn đơn giản (một câu đã chọn có thể bị khóa bởi câu chọn sau nó) — cần định nghĩa lại ràng buộc giữa mọi cặp.

In [28]:
def most_points_lock_before(questions):
    return Solution().mostPoints(questions[::-1])

def lock_before_brute(questions):
    n = len(questions); best = 0
    for mask in product([0, 1], repeat=n):
        ch = [i for i in range(n) if mask[i]]
        ok = all(not (i - questions[i][1] <= j < i) for i in ch for j in ch)
        if ok: best = max(best, sum(questions[i][0] for i in ch))
    return best

rng = random.Random(152)
for _ in range(400):
    q = [[rng.randint(1, 20), rng.randint(1, 3)] for _ in range(rng.randint(1, 9))]
    assert most_points_lock_before(q) == lock_before_brute(q)
print("biến thể khóa phía trước: đảo mảng là đủ — khớp brute force")

biến thể khóa phía trước: đảo mảng là đủ — khớp brute force


### 15.3 Biến thể C — Đếm **số cách chơi** đạt điểm tối đa (mod 10⁹+7)

Mang theo cặp `(điểm tốt nhất, số cách)` cho mỗi `dp[i]`; khi hai lựa chọn hòa thì cộng số cách. (Hai "cách chơi" khác nhau ⇔ khác tập câu được giải.)

In [29]:
def count_best_ways(questions, MOD=10**9 + 7):
    n = len(questions); best = [0] * (n + 1); ways = [1] * (n + 1)
    for i in range(n - 1, -1, -1):
        p, b = questions[i]; j = min(n, i + b + 1)
        s1, w1 = best[i + 1], ways[i + 1]
        s2, w2 = p + best[j], ways[j]
        if s1 > s2: best[i], ways[i] = s1, w1
        elif s2 > s1: best[i], ways[i] = s2, w2
        else: best[i], ways[i] = s1, (w1 + w2) % MOD
    return best[0], ways[0]

def count_best_brute(questions):
    n = len(questions); vals = []
    for mask in product([0, 1], repeat=n):
        s = score(questions, [i for i in range(n) if mask[i]])
        if s is not None: vals.append(s)
    m = max(vals); return m, vals.count(m)

print(count_best_ways([[1, 1], [1, 1], [1, 1]]), count_best_ways(examples[0][0]))
rng = random.Random(153)
for _ in range(400):
    q = [[rng.randint(1, 4), rng.randint(1, 3)] for _ in range(rng.randint(1, 9))]
    assert count_best_ways(q) == count_best_brute(q)
print("khớp brute force")

(2, 1) (5, 1)
khớp brute force


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi quyết định / tập con | O(2ⁿ) | O(n) | kiểm chứng |
| 2. Đệ quy có nhớ | `dfs(i)` + cache | O(n) | O(n) + stack | nháp ý tưởng, n nhỏ |
| 3. DP lùi | `dp[i] = max(dp[i+1], p + dp[i+b+1])` | O(n) | O(n) | **mặc định** |
| 4. DP đẩy | cập nhật `f[i+1]`, `f[i+b+1]` | O(n) | O(n) | thích duyệt xuôi |
| 5. DP + truy vết | lưu quyết định | O(n) | O(n) | cần in lời giải |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `mostPoints([[1,1],[2,1],[3,1],[4,1]])` và `mostPoints([[100,5],[1,1],[1,1]])`?

**Bài 2 (tính tay):** Tính mảng `dp` (đầy đủ) cho `[[2,1],[3,2],[1,1],[4,1],[2,3]]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(questions)` trả về **danh sách chỉ số câu được giải** trong một lời giải tối ưu (truy vết).

**Bài 4 (tự viết lại):** Viết `bt4_todo(questions)` cài DP đẩy (cách 4) không nhìn code.

**Bài 5 (mini challenge):** Mỗi câu có thêm **thời gian** `t_i` và bạn chỉ có tổng thời gian `T`. Viết `bt5_todo(questions, times, T)` trả về điểm tối đa (vừa luật brainpower vừa giới hạn thời gian). `n ≤ 50`, `T ≤ 100`.

In [30]:
def bt3_todo(questions):
    # TODO: danh sách câu được giải (truy vết)
    pass

def bt4_todo(questions):
    # TODO: DP đẩy
    pass

def bt5_todo(questions, times, T):
    # TODO: brainpower + giới hạn tổng thời gian
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: mỗi câu khóa 1 câu sau ⇒ House Robber. Mức 2: chọn câu 1 và 3 (2 + 4). Mức 3: `6`; ví dụ hai: giải câu 0 (100) khóa hết ⇒ `100`.

**Bài 2.** Mức 1: `n = 5`, `dp[5] = 0`. Mức 2: `dp[4] = 2`, `dp[3] = max(2, 4 + dp[5]) = 4`, `dp[2] = max(4, 1 + dp[4]) = 4`. Mức 3: `dp[1] = max(4, 3 + dp[4]) = 5`, `dp[0] = max(5, 2 + dp[2]) = 6` ⇒ `[6, 5, 4, 4, 2, 0]`.

**Bài 3.** Mức 1: lưu `take[i]`. Mức 2: đi xuôi từ 0, theo `take`. Mức 3: như `dp_with_choice`.

**Bài 4.** Mức 1: `f` dài `n + 1`, `f[0] = 0`. Mức 2: với mỗi `i`, đẩy sang `i + 1` (bỏ) và `min(n, i+b+1)` (giải). Mức 3: trả `f[n]`.

**Bài 5.** Mức 1: thêm chiều thời gian còn lại. Mức 2: `dp[i][r]` = điểm tối đa từ câu `i` với `r` đơn vị thời gian còn lại. Mức 3: `dp[i][r] = max(dp[i+1][r], p_i + dp[j][r − t_i] nếu r ≥ t_i)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `6`, `100`. **Bài 2:** `dp = [6, 5, 4, 4, 2, 0]`.

In [31]:
assert Solution().mostPoints([[1, 1], [2, 1], [3, 1], [4, 1]]) == 6
assert Solution().mostPoints([[100, 5], [1, 1], [1, 1]]) == 100
q = [[2, 1], [3, 2], [1, 1], [4, 1], [2, 3]]; n = len(q); dp = [0] * (n + 1)
for i in range(n - 1, -1, -1):
    p, b = q[i]; dp[i] = max(dp[i + 1], p + dp[min(n, i + b + 1)])
assert dp == [6, 5, 4, 4, 2, 0]

def bt3(questions):
    return dp_with_choice(questions)[1]

def bt4(questions):
    n = len(questions); f = [0] * (n + 1)
    for i, (p, b) in enumerate(questions):
        if f[i] > f[i + 1]: f[i + 1] = f[i]
        j = min(n, i + b + 1)
        if f[i] + p > f[j]: f[j] = f[i] + p
    return f[n]

def bt5(questions, times, T):
    n = len(questions)
    dp = [[0] * (T + 1) for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        p, b = questions[i]; j = min(n, i + b + 1); t = times[i]
        for r in range(T + 1):
            dp[i][r] = dp[i + 1][r]
            if r >= t: dp[i][r] = max(dp[i][r], p + dp[j][r - t])
    return dp[0][T]

def bt5_brute(questions, times, T):
    n = len(questions); best = 0
    for mask in product([0, 1], repeat=n):
        ch = [i for i in range(n) if mask[i]]
        s = score(questions, ch)
        if s is not None and sum(times[i] for i in ch) <= T: best = max(best, s)
    return best

rng = random.Random(1940)
for _ in range(1000):
    q = [[rng.randint(1, 20), rng.randint(1, 4)] for _ in range(rng.randint(1, 10))]
    ref = brute_subsets(q)
    ch = bt3(q)
    assert score(q, ch) == ref and bt4(q) == ref
    tms = [rng.randint(1, 5) for _ in q]; T = rng.randint(0, 12)
    assert bt5(q, tms, T) == bt5_brute(q, tms, T)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
n = len(questions)
dp = [0] * (n + 1)                       # dp[i]: tốt nhất từ câu i tới hết
for i in range(n - 1, -1, -1):
    p, b = questions[i]
    dp[i] = max(dp[i + 1],               # bỏ
                p + dp[min(n, i + b + 1)])  # giải: nhảy qua b câu
return dp[0]
```

**Checklist:**
- [ ] Nhảy tới `i + b + 1` (không phải `i + b`).
- [ ] Chặn `min(n, …)` hoặc mảng dài `n + 1` với `dp[n] = 0`.
- [ ] Duyệt **từ phải sang trái** (hoặc DP đẩy nếu đi xuôi).
- [ ] Không nộp đệ quy sâu `n = 10⁵`.
- [ ] (C++/Java) kiểu 64-bit cho tổng điểm.